[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-03-task-dependencies.ipynb#scrollTo=11223344)

---
# Day 3 · Task Dependencies and Data Passing
**certified-journeys / prefect-certified** · Day 3 · Learn

> **Goal for today:** Build multi-task flows where tasks pass data sequentially, submit tasks concurrently with `task.submit()` and `task.map()`, collect results with `PrefectFuture`, and use `wait_for` to express non-data dependencies.


In [ ]:
%pip install -q prefect


## Step 1 · Sequential data-passing — a 4-task linear chain

The most fundamental dependency pattern: each task receives the previous task's output as an argument. Prefect automatically enforces execution order because it can see the data flow.

```
fetch() → clean() → enrich() → report()
  |           |         |          |
 raw        cleaned  enriched   final
```

This is **synchronous by default** — Prefect runs each task to completion before starting the next, exactly like normal Python function calls. You get observability (state tracking, logs, retry) for free without changing your execution model.


In [ ]:
import json
import urllib.request
from datetime import datetime
from prefect import flow, task, get_run_logger

# ── Task 1: Fetch raw posts ───────────────────────────────────────────────────
@task(name="t1-fetch")
def fetch_posts(limit: int = 8) -> list:
    """Fetch raw posts from JSONPlaceholder (free mock API)."""
    url = f"https://jsonplaceholder.typicode.com/posts?_limit={limit}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

# ── Task 2: Clean — remove short posts ───────────────────────────────────────
@task(name="t2-clean")
def clean_posts(posts: list, min_body_words: int = 20) -> list:
    """Drop posts whose body is below the minimum word count."""
    return [p for p in posts if len(p["body"].split()) >= min_body_words]

# ── Task 3: Enrich — add a word-count field ───────────────────────────────────
@task(name="t3-enrich")
def enrich_posts(posts: list) -> list:
    """Add a word_count field to each post record."""
    return [{**p, "word_count": len(p["body"].split())} for p in posts]

# ── Task 4: Report — aggregate metrics ───────────────────────────────────────
@task(name="t4-report")
def build_report(posts: list) -> dict:
    """Produce a final summary report from the enriched posts."""
    counts = [p["word_count"] for p in posts]
    return {
        "n_posts":     len(posts),
        "avg_words":   round(sum(counts) / len(counts), 1) if counts else 0,
        "max_words":   max(counts, default=0),
        "generated_at": datetime.utcnow().isoformat(),
    }

# ── Flow: thin orchestrator — just chains tasks ───────────────────────────────
@flow(name="sequential-chain", log_prints=True)
def sequential_chain(limit: int = 8, min_words: int = 20):
    logger = get_run_logger()

    raw      = fetch_posts(limit)            # Task 1 → returns list
    cleaned  = clean_posts(raw, min_words)   # Task 2 → receives Task 1's output
    enriched = enrich_posts(cleaned)          # Task 3 → receives Task 2's output
    report   = build_report(enriched)         # Task 4 → receives Task 3's output

    logger.info(f"Report: {report}")
    print(f"Kept {report['n_posts']}/{limit} posts, avg {report['avg_words']} words")
    return report

result = sequential_chain(limit=10, min_words=25)
print(f"\nFinal report: {result}")


### What just happened?

- Four task runs were created and executed in strict sequence: `t1-fetch → t2-clean → t3-enrich → t4-report`.
- **Each task's return value was passed directly as an argument to the next** — that single pattern is everything Prefect needs to enforce ordering.
- If `t2-clean` had failed, `t3-enrich` and `t4-report` would never have run — their task runs would not appear in the UI at all.
- In the Prefect UI Graph view this appears as a straight left-to-right line of four connected nodes.


## Step 2 · `task.submit()` and `PrefectFuture` — non-blocking concurrent execution

By default, calling `my_task(args)` **blocks** until the task completes and returns its result. Use `my_task.submit(args)` instead to submit the task without blocking, receiving a **`PrefectFuture`** immediately.

| Calling style | Blocks? | Returns | Use when |
|---|---|---|---|
| `my_task(args)` | Yes | Python result directly | Tasks must run strictly in order |
| `my_task.submit(args)` | No | `PrefectFuture` | Tasks can run concurrently |
| `future.result()` | Yes | Python result | You need the value from a future |

`PrefectFuture` is essentially a handle to an in-flight or completed task run. Calling `.result()` blocks until the task finishes and unwraps the return value.

**Important:** Prefect still respects data dependencies when you use `.submit()`. If you pass a future as an argument to another `.submit()` call, Prefect waits for the first future to resolve before starting the second.


In [ ]:
import json
import urllib.request
import time
from prefect import flow, task
from prefect.futures import PrefectFuture

@task(name="fetch-user")
def fetch_user(user_id: int) -> dict:
    """Fetch a single user profile — simulates a slow I/O call."""
    time.sleep(0.3)  # simulate latency; in production this is a DB or API call
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@task(name="format-user")
def format_user(user: dict) -> str:
    """Format user into a display string."""
    return f"{user['name']} <{user['email']}> @ {user['company']['name']}"

@flow(name="concurrent-users", log_prints=True)
def concurrent_users():
    """Submit 5 fetch tasks concurrently, then collect results."""
    start = time.perf_counter()

    # Submit all fetches at once — each returns a PrefectFuture immediately
    futures: list[PrefectFuture] = [
        fetch_user.submit(user_id)          # non-blocking — returns future right away
        for user_id in range(1, 6)
    ]
    print(f"All 5 fetch tasks submitted after {time.perf_counter()-start:.2f}s")

    # Block to collect results — waits for each task to complete
    users = [f.result() for f in futures]   # .result() blocks until task finishes

    elapsed = time.perf_counter() - start
    print(f"All 5 fetches completed in {elapsed:.2f}s (sequential would be ~1.5s)")

    # Now format each user — these can also run concurrently
    format_futures = [format_user.submit(u) for u in users]
    display_lines  = [f.result() for f in format_futures]

    for line in display_lines:
        print(f"  {line}")

    return display_lines

lines = concurrent_users()
print(f"\nFormatted {len(lines)} users")


### What just happened?

- **`fetch_user.submit(user_id)`** returned a `PrefectFuture` without waiting for the task to finish — all 5 submits happened nearly instantly.
- **`f.result()`** blocked until each task completed and returned the Python value.
- The 5 fetch tasks ran **concurrently** — total wall time was roughly one task's latency, not five.
- In the Prefect UI Timeline view (Runs → this run → Timeline), you can see all 5 `fetch-user` task bars overlapping, confirming they ran in parallel.


## Step 3 · `task.map()` — submit over an iterable in one call

`task.map(iterable)` is shorthand for `[task.submit(item) for item in iterable]`. It returns a list of `PrefectFuture` objects and is the idiomatic way to fan out over a list of inputs.

| | `.submit()` loop | `.map()` |
|---|---|---|
| Syntax | `[task.submit(x) for x in items]` | `task.map(items)` |
| Returns | `list[PrefectFuture]` | `list[PrefectFuture]` |
| Concurrency | Yes | Yes |
| Static parameters | Pass directly | Use `unmapped()` for non-iterable args |

Use `unmapped(value)` to pass a constant argument that should NOT be iterated — for example, a shared configuration dict or an API key.


In [ ]:
import json
import urllib.request
from prefect import flow, task, unmapped
from prefect.futures import PrefectFuture

@task(name="fetch-post")
def fetch_post(post_id: int) -> dict:
    """Fetch a single post by ID."""
    url = f"https://jsonplaceholder.typicode.com/posts/{post_id}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@task(name="score-post")
def score_post(post: dict, max_score: int) -> dict:
    """
    Score a post by body word count, normalised to max_score.
    max_score is a constant — use unmapped() so it isn't iterated.
    """
    words = len(post["body"].split())
    score = min(round(words / 30 * max_score), max_score)  # cap at max_score
    return {"id": post["id"], "title": post["title"][:40], "score": score}

@flow(name="map-demo", log_prints=True)
def map_demo(post_ids: list = None, max_score: int = 100):
    """Fetch and score multiple posts using .map() for concurrent fan-out."""
    if post_ids is None:
        post_ids = list(range(1, 7))  # posts 1–6

    # Fan out: fetch all posts concurrently — one future per post_id
    post_futures: list[PrefectFuture] = fetch_post.map(post_ids)

    # score_post takes (post, max_score) — post is mapped, max_score is constant
    # unmapped() tells Prefect: do NOT iterate over max_score
    score_futures: list[PrefectFuture] = score_post.map(
        post_futures,           # Prefect resolves futures automatically before passing
        unmapped(max_score),    # same max_score value sent to every task
    )

    # Collect all scored results
    scores = [f.result() for f in score_futures]

    # Sort by score descending
    scores.sort(key=lambda x: x["score"], reverse=True)
    for s in scores:
        print(f"  [{s['score']:3d}] {s['title']}")

    return scores

scores = map_demo(post_ids=list(range(1, 9)), max_score=100)
print(f"\nTop post: [{scores[0]['score']}] {scores[0]['title']}")


### What just happened?

- **`fetch_post.map(post_ids)`** submitted one task per `post_id` in the list and returned a list of futures — a single line replaced an explicit comprehension.
- **`score_post.map(post_futures, unmapped(max_score))`** shows two important behaviours: futures passed directly are resolved automatically (Prefect waits for the fetch to finish before scoring), and `unmapped()` prevents `max_score` from being iterated.
- All 8 fetches ran concurrently, all 8 score tasks ran concurrently once their upstream fetch resolved — 16 task runs in total.
- In the UI Timeline, you should see two distinct horizontal bands: all fetch tasks overlapping, then all score tasks overlapping shortly after.


## Step 4 · Collecting futures with `.result()` — blocking strategies

When you have a list of futures, there are several ways to collect results. The choice affects when your flow blocks and how errors surface.

| Pattern | Blocks until | Error behaviour |
|---|---|---|
| `[f.result() for f in futures]` | Each future in order | Raises on first failure |
| `[f.result(raise_on_failure=False) for f in futures]` | Each future in order | Returns `State` objects for failures |
| `PrefectFuture.wait(futures)` | All complete | Then inspect states |

Use `raise_on_failure=False` when you want to process partial results — for example, 9 out of 10 API calls succeeded and you don't want to discard the 9 successes because of 1 failure.


In [ ]:
import json
import urllib.request
from prefect import flow, task
from prefect.futures import PrefectFuture

_attempt_tracker = {}  # track which task IDs have been attempted

@task(name="flaky-post-fetch", retries=0)  # retries=0 so failures are immediate
def flaky_post_fetch(post_id: int) -> dict:
    """Fetch a post, but intentionally fail for even post_ids to demo error handling."""
    if post_id % 2 == 0:  # even IDs fail
        raise ValueError(f"Simulated failure for post_id={post_id}")
    url = f"https://jsonplaceholder.typicode.com/posts/{post_id}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@flow(name="partial-results-flow", log_prints=True)
def partial_results_flow():
    """
    Show how to collect partial results when some tasks fail.
    Posts 1,3,5 will succeed; posts 2,4 will fail.
    """
    post_ids = [1, 2, 3, 4, 5]
    futures: list[PrefectFuture] = flaky_post_fetch.map(post_ids)

    successes = []
    failures  = []

    for post_id, future in zip(post_ids, futures):
        # raise_on_failure=False returns the State object instead of raising
        result = future.result(raise_on_failure=False)
        if hasattr(result, "is_failed") and result.is_failed():
            failures.append(post_id)
            print(f"  FAILED  post_id={post_id}")
        else:
            # result is the actual dict when the task succeeded
            successes.append(result)
            print(f"  OK      post_id={post_id} — '{result['title'][:35]}'")

    print(f"\nSucceeded: {len(successes)}/{len(post_ids)}, Failed: {len(failures)}/{len(post_ids)}")
    return {"successes": len(successes), "failures": failures}

summary = partial_results_flow()
print(f"\nSummary: {summary}")


### What just happened?

- **`raise_on_failure=False`** is the key: instead of raising an exception for failed tasks, `.result()` returns the `State` object.
- We check `result.is_failed()` to distinguish between a successful return value (a dict) and a failure state.
- The flow **completed successfully** even though 2 of its 5 tasks failed — because we handled the failures explicitly.
- In production, you'd log the `failures` list to a monitoring system or dead-letter queue and re-process them later.


## Step 5 · `wait_for` — non-data dependencies between tasks

Sometimes you need task B to run after task A even though B doesn't use A's return value. This is a **non-data dependency** — you can't express it by passing a return value.

Common cases:
- A cleanup task must run after a load task (but doesn't need the loaded data)
- A notification task must run after all processing tasks (but only needs their count, not their data)
- A schema migration must complete before any write tasks start

Use the **`wait_for`** parameter to express this:

```python
cleanup_future = cleanup.submit(wait_for=[load_future])
```

Prefect will hold `cleanup` until `load` completes, without requiring any data to flow between them.


In [ ]:
import json
import urllib.request
import time
from prefect import flow, task
from prefect.futures import PrefectFuture

@task(name="ingest-posts")
def ingest_posts(limit: int = 5) -> int:
    """Simulate ingesting posts — returns the count of records written."""
    url = f"https://jsonplaceholder.typicode.com/posts?_limit={limit}"
    with urllib.request.urlopen(url) as resp:
        posts = json.loads(resp.read())
    time.sleep(0.2)  # simulate write latency
    return len(posts)

@task(name="update-index")
def update_index(limit: int = 5) -> int:
    """Simulate updating a search index — also writes records."""
    url = f"https://jsonplaceholder.typicode.com/todos?_limit={limit}"
    with urllib.request.urlopen(url) as resp:
        todos = json.loads(resp.read())
    time.sleep(0.2)  # simulate index write latency
    return len(todos)

@task(name="send-notification", log_prints=True)
def send_notification(message: str) -> str:
    """
    Send a completion notification.
    This task does NOT need the return values of ingest_posts or update_index
    — it only needs to know they finished successfully.
    """
    print(f"[NOTIFY] {message}")
    return "notification_sent"

@task(name="archive-logs", log_prints=True)
def archive_logs(source: str) -> str:
    """Archive run logs — must wait for notification before archiving."""
    print(f"[ARCHIVE] Archiving logs for source: {source}")
    return "archived"

@flow(name="wait-for-demo", log_prints=True)
def wait_for_demo():
    """
    Demonstrate wait_for:
      ingest_posts ─┐
                    ├──► send_notification ──► archive_logs
      update_index ─┘
    send_notification doesn't use the return values of ingest/update — pure ordering.
    """
    # Submit both ingestion tasks concurrently
    ingest_future = ingest_posts.submit(limit=5)
    index_future  = update_index.submit(limit=5)

    # send_notification doesn't need the counts — just needs both to finish
    # wait_for accepts a list of futures; Prefect blocks until all are resolved
    notify_future = send_notification.submit(
        "Pipeline stage 1 complete — ingest and index tasks finished.",
        wait_for=[ingest_future, index_future],  # non-data dependency
    )

    # archive_logs also uses wait_for — must follow the notification
    archive_future = archive_logs.submit(
        "pipeline-run-today",
        wait_for=[notify_future],                # non-data dependency
    )

    # Collect final results
    n_posts = ingest_future.result()
    n_todos = index_future.result()
    notify_future.result()
    archive_status = archive_future.result()

    print(f"Ingested {n_posts} posts, indexed {n_todos} todos")
    print(f"Archive status: {archive_status}")

wait_for_demo()


### What just happened?

- `ingest_posts` and `update_index` ran **concurrently** — they were both submitted without `wait_for`.
- `send_notification` had **no data connection** to either ingest task, but `wait_for=[ingest_future, index_future]` told Prefect to hold it until both completed.
- `archive_logs` then used `wait_for=[notify_future]` to chain after the notification — again, no data flows between them.
- In the UI Graph view, you'll see edges from `ingest_posts` → `send_notification` and `update_index` → `send_notification`, even though no Python values pass along those edges — these are `wait_for` dependency edges.


## Step 6 · Parallel vs sequential — observing the timeline difference

Running the same workload sequentially (default call) vs concurrently (`.submit()`) produces visibly different timeline charts in the Prefect UI.

The code below runs both versions and measures wall time so you can see the difference numerically. In the UI you'll see the difference visually: sequential runs stack end-to-end, concurrent runs overlap.


In [ ]:
import json
import urllib.request
import time
from prefect import flow, task
from prefect.futures import PrefectFuture

@task(name="slow-fetch")
def slow_fetch(user_id: int) -> dict:
    """Simulates a slow per-user API call (0.4 s latency)."""
    time.sleep(0.4)  # simulate I/O latency; production: DB query, REST call, etc.
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

@flow(name="sequential-version", log_prints=True)
def sequential_version(user_ids: list = None) -> list:
    """Call tasks one at a time — total time ≈ N × latency."""
    if user_ids is None:
        user_ids = list(range(1, 5))  # 4 users
    start = time.perf_counter()
    results = [slow_fetch(uid) for uid in user_ids]  # blocking call each time
    elapsed = time.perf_counter() - start
    print(f"Sequential: {len(results)} users in {elapsed:.2f}s")
    return results

@flow(name="concurrent-version", log_prints=True)
def concurrent_version(user_ids: list = None) -> list:
    """Submit tasks concurrently — total time ≈ 1 × latency."""
    if user_ids is None:
        user_ids = list(range(1, 5))  # same 4 users
    start = time.perf_counter()
    futures: list[PrefectFuture] = [slow_fetch.submit(uid) for uid in user_ids]  # non-blocking
    results = [f.result() for f in futures]  # collect after all are submitted
    elapsed = time.perf_counter() - start
    print(f"Concurrent: {len(results)} users in {elapsed:.2f}s")
    return results

# Run both and compare
print("=== Sequential ===")
seq_results = sequential_version()

print("\n=== Concurrent ===")
con_results = concurrent_version()

print("\nBoth returned the same users:")
for s, c in zip(seq_results, con_results):
    assert s["id"] == c["id"], "IDs should match"
    print(f"  User {s['id']}: {s['name']}")
print("\nOpen the UI to compare the Timeline tabs of the two flow runs.")
print("Sequential: task bars are end-to-end. Concurrent: task bars overlap.")


### What just happened?

- The **sequential** version took roughly `N × 0.4 s` — each task waited for the previous one.
- The **concurrent** version took roughly `0.4 s` regardless of N — all tasks ran in parallel.
- **Both produced identical results** — the order of `.result()` collection was sequential (by index), even though execution was parallel.
- In the Prefect UI **Timeline view**, sequential runs show non-overlapping task bars; concurrent runs show all task bars starting at nearly the same time and overlapping.
- Use `task.submit()` whenever tasks are I/O-bound (HTTP, database, file) and don't depend on each other's data — the speedup is proportional to the number of independent tasks.


In [ ]:
# Challenge: Build a pipeline that mixes sequential and concurrent patterns
#
# Requirements:
#   1. Task `fetch_album(album_id)` — fetch album from
#      https://jsonplaceholder.typicode.com/albums/{album_id}
#      (returns dict with keys: id, userId, title)
#
#   2. Task `fetch_photos(album_id)` — fetch photos for an album from
#      https://jsonplaceholder.typicode.com/photos?albumId={album_id}
#      (returns list of photo dicts)
#
#   3. Task `summarise_album(album, photos)` — receives BOTH album and photos,
#      returns dict: {album_id, title, photo_count, user_id}
#
#   4. Task `write_report(summaries)` — receives a list of summary dicts,
#      prints them, returns total photo count across all albums
#
#   5. Flow `album_pipeline(album_ids=[1,2,3])` that:
#      a. Fetches all albums AND photos concurrently (use .submit() or .map())
#      b. Summarises each album (one summarise task per album)
#      c. Calls write_report AFTER all summaries complete
#         (use wait_for if write_report doesn't receive data from summaries,
#          or just pass the collected list directly if it does)
#
# Scaffold:

import json
import urllib.request
from prefect import flow, task
from prefect.futures import PrefectFuture

# TODO: implement Task 1 — fetch_album
# @task(name="fetch-album")
# def fetch_album(album_id: int) -> dict:
#     ...

# TODO: implement Task 2 — fetch_photos
# @task(name="fetch-photos")
# def fetch_photos(album_id: int) -> list:
#     ...

# TODO: implement Task 3 — summarise_album
# @task(name="summarise-album")
# def summarise_album(album: dict, photos: list) -> dict:
#     ...

# TODO: implement Task 4 — write_report
# @task(name="write-report", log_prints=True)
# def write_report(summaries: list) -> int:
#     ...

# TODO: implement the flow
# @flow(name="album-pipeline", log_prints=True)
# def album_pipeline(album_ids: list = None):
#     ...

# Uncomment once implemented:
# album_pipeline(album_ids=[1, 2, 3])


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| Sequential chain | Pass return values between tasks — Prefect infers dependency order automatically |
| `task.submit()` | Non-blocking — returns `PrefectFuture` immediately; tasks run concurrently |
| `PrefectFuture` | Handle to an in-flight task run; call `.result()` to block and get the value |
| `raise_on_failure=False` | `.result()` returns a `State` object instead of raising — enables partial-results handling |
| `task.map(iterable)` | Shorthand for `[task.submit(x) for x in iterable]` — fan-out over a list |
| `unmapped(value)` | Use with `.map()` to pass a constant argument that should NOT be iterated |
| `wait_for=[futures]` | Express non-data ordering: task B runs after task A with no value passing |
| Timeline view | Runs → [flow run] → Timeline — see sequential vs concurrent task bars visually |

> **Tip:** Use `task.submit()` instead of calling tasks directly when you want non-blocking, concurrent execution — Prefect will respect data-dependency order automatically.

---
## What's next
**Day 4** → Add retries and caching to tasks — configure `retry_condition_fn` for smarter retry logic and `cache_key_fn` to skip expensive re-computation on re-runs.

Mark Day 3 complete in your [tracker](../index.html).
